<a href="https://colab.research.google.com/github/ShuAng1602/ADALL_Github/blob/main/CBA1C03_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip -q install -U \
  scikit-learn==1.8.0
# Most stable version

In [8]:
# Core libraries
import pandas as pd
import numpy as np
# Visualisation
import matplotlib.pyplot as plt
# Modelling and preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, ShuffleSplit, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.inspection import permutation_importance

import xgboost as xgb
import sklearn

RANDOM_STATE = 42
print("Setup complete")

Setup complete


In [9]:
import sklearn
print("scikit-learn version:", sklearn.__version__)


scikit-learn version: 1.8.0


In [10]:
github_raw_url = 'https://raw.githubusercontent.com/ShuAng1602/ADALL_Github/refs/heads/main/fraud_oracle.csv'
try:
    df = pd.read_csv(github_raw_url) # index_col=0) #when data is starting with index so start with 0
    print("Successfully loaded data from GitHub!")
    display(df.head()) # show top 5
except Exception as e:
    print(f"Error loading data: {e}")
    print("Please ensure the URL is correct and the file format is compatible with `pd.read_csv`.")

Successfully loaded data from GitHub!


,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision


In [11]:
df.columns

Index(['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea',
       'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed', 'Sex',
       'MaritalStatus', 'Age', 'Fault', 'PolicyType', 'VehicleCategory',
       'VehiclePrice', 'FraudFound_P', 'PolicyNumber', 'RepNumber',
       'Deductible', 'DriverRating', 'Days_Policy_Accident',
       'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle',
       'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType',
       'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'Year',
       'BasePolicy'],
      dtype='object')

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Month                 15420 non-null  object
 1   WeekOfMonth           15420 non-null  int64 
 2   DayOfWeek             15420 non-null  object
 3   Make                  15420 non-null  object
 4   AccidentArea          15420 non-null  object
 5   DayOfWeekClaimed      15420 non-null  object
 6   MonthClaimed          15420 non-null  object
 7   WeekOfMonthClaimed    15420 non-null  int64 
 8   Sex                   15420 non-null  object
 9   MaritalStatus         15420 non-null  object
 10  Age                   15420 non-null  int64 
 11  Fault                 15420 non-null  object
 12  PolicyType            15420 non-null  object
 13  VehicleCategory       15420 non-null  object
 14  VehiclePrice          15420 non-null  object
 15  FraudFound_P          15420 non-null

In [13]:
from google.colab import userdata
from openai import OpenAI

# Load key from Google Colab Secrets
api_key = userdata.get('OPENAI_API_KEY')

client = OpenAI(
    api_key=api_key,
)

In [18]:
import pandas as pd
import numpy as np
from io import StringIO
# ---------------------------
# Generate a full dataset profile
# ---------------------------
buffer = StringIO()

# dtypes
buffer.write("=== DTYPES ===\n")
buffer.write(df.dtypes.to_string())
buffer.write("\n\n")

# numeric describe
buffer.write("=== NUMERIC DESCRIBE ===\n")
buffer.write(df.describe().to_string())
buffer.write("\n\n")

# categorical describe
buffer.write("=== CATEGORICAL DESCRIBE ===\n")
try:
    buffer.write(df.describe(include='object').to_string())
except:
    buffer.write("No categorical columns")
buffer.write("\n\n")

# null summary
buffer.write("=== NULL SUMMARY ===\n")
null_summary = (
    df.isna().sum().to_frame("null_count")
    .assign(null_pct=lambda x: x["null_count"]/len(df))
)
buffer.write(null_summary.to_string())
buffer.write("\n\n")

# unique cardinality
buffer.write("=== UNIQUE VALUES PER COLUMN ===\n")
buffer.write(df.nunique().to_frame("unique_count").to_string())
buffer.write("\n\n")

# correlation matrix
buffer.write("=== CORRELATIONS (NUMERIC ONLY) ===\n")
buffer.write(df.corr(numeric_only=True).round(3).to_string())
buffer.write("\n\n")

# value counts for categoricals
buffer.write("=== VALUE COUNTS (TOP 20 PER CATEGORICAL COLUMN) ===\n")
cat_cols = df.select_dtypes(include='object').columns
if len(cat_cols) > 0:
    for col in cat_cols:
        buffer.write(f"\nColumn: {col}\n")
        vc = df[col].value_counts().head(20)
        buffer.write(vc.to_string())
        buffer.write("\n")
else:
    buffer.write("No categorical columns\n")
buffer.write("\n")

# --------- FIXED OUTLIER COMPUTATION (NO BOOLEANS) ---------
buffer.write("=== OUTLIER SUMMARY (IQR METHOD) ===\n")
num_cols = df.select_dtypes(include=['number']).columns  # exclude booleans
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1
outliers = ((df[num_cols] < (Q1 - 1.5*IQR)) | (df[num_cols] > (Q3 + 1.5*IQR))).sum()
buffer.write(outliers.to_string())
buffer.write("\n\n")

# leakage scan: columns with all unique values
buffer.write("=== POSSIBLE LEAKAGE COLUMNS (UNIQUE FOR EACH ROW) ===\n")
leak_cols = df.columns[df.nunique() == len(df)]
buffer.write(str(list(leak_cols)))
buffer.write("\n\n")

# shape, duplicates, constant cols
buffer.write("=== SHAPE / DUPLICATES / CONSTANT COLUMNS ===\n")
dup_count = df.duplicated().sum()
constant_cols = df.columns[df.nunique() == 1].tolist()
buffer.write(f"Rows: {len(df)}, Columns: {df.shape[1]}\n")
buffer.write(f"Duplicate rows: {dup_count}\n")
buffer.write(f"Constant columns: {constant_cols}\n\n")

# Final text
payload_text = buffer.getvalue()

print(payload_text)

=== DTYPES ===
Month                   object
WeekOfMonth              int64
DayOfWeek               object
Make                    object
AccidentArea            object
DayOfWeekClaimed        object
MonthClaimed            object
WeekOfMonthClaimed       int64
Sex                     object
MaritalStatus           object
Age                      int64
Fault                   object
PolicyType              object
VehicleCategory         object
VehiclePrice            object
FraudFound_P             int64
PolicyNumber             int64
RepNumber                int64
Deductible               int64
DriverRating             int64
Days_Policy_Accident    object
Days_Policy_Claim       object
PastNumberOfClaims      object
AgeOfVehicle            object
AgeOfPolicyHolder       object
PoliceReportFiled       object
WitnessPresent          object
AgentType               object
NumberOfSuppliments     object
AddressChange_Claim     object
NumberOfCars            object
Year                    

In [19]:
# sending to LLM API
response = client.responses.create(
    model="gpt-5-mini",
    instructions="""
You are an expert data scientist specialising in insurance analytics and fraud detection.
Use ONLY the information inside the provided dataset profile text.
Do NOT invent correlations, columns, distributions, or values.
If a detail is not explicitly present in the dataset profile, clearly state: 'Not shown in profile'.
Always justify reasoning and recommendations based ONLY on the dataset profile and business context.
Focus on clarity, business relevance, and modelling alignment.
""",
    input=f"""
    Dataset info: {payload_text}\n

    Context:
    The business problem relates to detecting fraudulent vehicle insurance claims.
    The dataset contains vehicle details, accident attributes, and insurance policy information.
    The target variable is FraudFound_P, where 1 indicates a fraudulent claim and 0 indicates a legitimate claim.

    Questions:
    1. Describe the business problem you aim to address. Explain why it matters and who is affected.

    2. Identify the key beneficiary of the ML solution. Create a simple persona that captures:
      - Goals
      - Pain points
      - Behaviours

    3. Use the Job-To-Be-Done (JTBD) framework to shape your modelling objectives.
      - What job is the user trying to complete?
      - What is the most appropriate target variable?
    """
    )

print(response.output_text)

1) Business problem — description, importance, affected parties
- Problem: Identify fraudulent vehicle insurance claims (binary outcome FraudFound_P: 1 = fraud, 0 = legitimate) so claims that are likely fraudulent can be triaged/investigated.
- Why it matters (justified from the profile):
  - Financial exposure: the dataset size (15,420 claims) and a non‑zero fraud prevalence (mean FraudFound_P = 0.059857 → ~6% of claims) imply a non‑trivial absolute number of fraudulent claims (≈ 900+ cases). Undetected fraud scales with volume and increases losses and premiums.
  - Operational load: there is a large volume of everyday claims to review (dataset contains many categorical policy and claim attributes that would be used to triage). Without automation, investigative teams face high manual workload.
  - Customer & regulatory impact: inaccurate handling (false positives or delayed legitimate payments) damages customer experience and risks regulatory/artifact costs.
- Who is affected:
  - Ins

In [20]:
# sending to LLM API
response = client.responses.create(
    model="gpt-5-mini",
    instructions="""
You are an expert data scientist specialising in insurance analytics and fraud detection.

Follow these rules strictly:
1. Use ONLY the information explicitly present in the provided dataset profile text.
2. Do NOT invent correlations, costs, KPIs, operational processes, or columns.
3. If something is not explicitly stated in the dataset profile, label it clearly as: 'Not shown in profile'.
4. Separate clearly between:
   - What is supported by the dataset profile
   - What is reasonable business context (label clearly)
5. Ensure all predictors mentioned are available BEFORE the fraud outcome is known
   (avoid future or post-investigation information).
6. Frame answers at a business decision level (who decides, what changes with a model).
7. Keep answers concise, structured, and suitable for a technical audience.

Your goal is to produce clear, defensible answers that would make sense to a business user.
""",
    input=f"""
Dataset info: {payload_text}\n

Context:
This dataset concerns vehicle insurance claims with the objective of detecting fraudulent claims.
The target variable is FraudFound_P (1 = fraudulent claim, 0 = legitimate claim).
The dataset contains claim-level records with vehicle, accident, policy, and claimant attributes.

Answer the following questions clearly and separately.

--------------------------------
1.1 Business Problem
--------------------------------
Describe the business problem addressed by this dataset.

Focus explicitly on:
- Who is affected (specific roles or teams, not generic users)
- Why the problem matters, using evidence from the dataset profile where possible
  (e.g. dataset size, fraud prevalence, claim volume, operational scale)
- What decision would change if a predictive model were available
  (e.g. which claims to investigate, how to prioritise workload)

Important constraints:
- Do NOT claim monetary savings unless costs are shown in the profile.
- If impacts (e.g. regulatory risk, customer dissatisfaction) are general industry context,
  state that they are 'Not shown in profile'.

--------------------------------
1.2 Persona
--------------------------------
Identify the key beneficiary of the ML solution and create ONE simple persona.

Focus on:
- A role that can realistically take action using model output
- What this persona does on a daily or weekly basis
- Their concrete goals and pain points, grounded in the dataset characteristics
  (e.g. high claim volume, low fraud prevalence, categorical-heavy data)
- What output they need from the model
  (e.g. ranked list of claims, risk score, supporting feature indicators)

Constraints:
- Avoid assuming dashboards, automation pipelines, or retraining workflows
  unless they are explicitly supported by the dataset (otherwise mark as 'Not shown in profile').

--------------------------------
1.3 JTBD Framework
--------------------------------
Use the Job-To-Be-Done (JTBD) framework to define the modelling objective.

Structure your answer as follows:
1. JTBD statement:
   - A concrete sentence describing the job the persona is trying to complete
     (focus on decision-making under constraints, not technical modelling).
2. Target variable:
   - Identify the most appropriate target variable from the dataset and justify why it matches the job.
3. Likely predictors:
   - Group predictors into logical categories (e.g. timing, policy, accident, vehicle).
   - Only include variables explicitly present in the dataset profile.
   - Ensure predictors are available before the fraud outcome is known.
   - If a column risks data leakage (e.g. identifiers), flag it clearly and explain why.

Do NOT:
- Introduce engineered features unless they are direct transformations of existing columns
  (and label them as 'derived').
- Assume cost functions, thresholds, or operational KPIs unless shown in the profile.

--------------------------------
End of questions.
"""
)

print(response.output_text)

1.1 Business Problem

Supported by dataset profile
- Problem statement: Use claim-level data to detect fraudulent vehicle-insurance claims (target = FraudFound_P, 1 = fraud, 0 = legitimate).
- Who is affected:
  - Fraud investigators / fraud detection team — they review flagged claims for investigation (dataset is explicitly about fraud detection).
  - Claims triage / claims operations managers — they decide which claims are escalated or prioritized.
  - Risk/underwriting analytics team — they would consume model outputs to understand emerging patterns.
- Why the problem matters (evidence in the profile):
  - Dataset size / operational scale: 15,420 claim records (rows = 15,420).
  - Fraud prevalence: mean(FraudFound_P) = 0.059857 → ~6% of claims labeled fraudulent (class imbalance).
  - High categorical dimensionality and many policy/vehicle/accident attributes (33 columns), indicating many decision inputs for triage.
- What decision would change with a predictive model:
  - Which cla

In [21]:
response = client.responses.create(
    model="gpt-5-mini",
    instructions="""
You are a senior data scientist specializing in tree-based models and data quality assessment.
Always justify recommendations using reasoning trace based ONLY on the dataset profile.

Rules:
- Base all analysis strictly on the provided dataset profile.
- Do NOT infer or assume information not explicitly stated.
- If information is missing, explicitly flag it as a limitation.
- Justifications must be concise, technical, and data-centric.
""",
    input=f"""
Dataset info: {payload_text}\n

Task:

SECTION 1 — Data Quality Assessment
Based on the dataset profile, identify all data quality issues that should be addressed BEFORE modelling.
- Provide a prioritized list (High / Medium / Low).
- For each issue:
  - Affected columns
  - Why it matters for tree-based models
  - Potential downstream impact if unresolved
  - Assess whether each recommendation is necessary at this stage and briefly justify agreement or disagreement.
Sample response:
Any transforming predictors required to uncover useful patterns.
Balancing imbalanced classes.
Feature combination or reduction is unnecessary at this stage, as XGBoost handles correlated predictors well.
Outlier removal is premature, since extreme values (e.g. older students) may represent meaningful signal rather than noise.

SECTION 2 — Feature Risk Analysis
Identify columns that are:
- Redundant
- Highly correlated
- Potential sources of data leakage

For each identified column:
- Explain the risk type
- Justify using only the dataset profile
- State whether it should be dropped, reviewed, or conditionally retained

SECTION 3 — Remediation Code
Provide a clean, reusable Python script that:
1. Defines one helper function per identified issue.
   - Each function must:
     - Accept a pandas DataFrame
     - Return a modified DataFrame
     - Contain a clear docstring
2. Defines a single wrapper function:
   - Accepts boolean flags to enable/disable each helper
3. Includes ONE single-line example showing how to run the wrapper.

Constraints:
- Do NOT encode categorical variables.
- Do NOT train or fit any model.
- Output Python code in a single fenced code block.
"""
)
print(response.output_text)

SECTION 1 — Data Quality Assessment (prioritized)

High priority
1) Target class imbalance
- Affected columns: FraudFound_P
- Why it matters: Positive class prevalence ≈ 5.99% (mean = 0.059857). Tree-based learners will learn majority class easily and may underperform on the rare positive class unless sampling or class-weighting is considered.
- Downstream impact if unresolved: Poor recall/precision on fraud class; misleading global accuracy; unreliable decision thresholds.
- Recommendation necessary now? Yes — necessary to plan modelling strategy (resampling or class-weights) before optimizing models.

2) Unique ID / leakage risk
- Affected columns: PolicyNumber
- Why it matters: PolicyNumber is unique for every row (unique_count = 15420). Unique identifiers can leak row-level identity and spuriously boost model performance.
- Downstream impact if unresolved: Overfitting and inflated performance in cross-validation; models may learn non-generalizable ID patterns.
- Recommendation nece

In [ ]:
import pandas as pd
import numpy as np

def drop_policy_number(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop the unique identifier column 'PolicyNumber' if present.
    Rationale: Profile shows PolicyNumber is unique per row and flagged as possible leakage.
    Returns a modified copy of df with PolicyNumber removed.
    """
    df = df.copy()
    if 'PolicyNumber' in df.columns:
        df = df.drop(columns=['PolicyNumber'])
    return df

def replace_placeholder_zeros(df: pd.DataFrame, cols=None) -> pd.DataFrame:
    """
    Replace literal string '0' entries in specified categorical columns with NaN.
    Default targets columns flagged in the profile: 'DayOfWeekClaimed' and 'MonthClaimed'.
    Returns modified copy of df.
    """
    df = df.copy()
    if cols is None:
        cols = ['DayOfWeekClaimed', 'MonthClaimed']
    for c in cols:
        if c in df.columns:
            # only replace exact string '0' (not integer 0)
            df[c] = df[c].replace('0', np.nan)
    return df

def convert_object_to_category(df: pd.DataFrame, exclude=None) -> pd.DataFrame:
    """
    Convert object-dtype columns to pandas 'category' dtype to reduce memory and mark them as categorical.
    Does NOT encode categories; simply changes dtype.
    exclude: optional list of columns to leave as object.
    Returns modified copy of df.
    """
    df = df.copy()
    if exclude is None:
        exclude = []
    obj_cols = df.select_dtypes(include=['object']).columns.tolist()
    for c in obj_cols:
        if c in exclude:
            continue
        df[c] = df[c].astype('category')
    return df

def handle_age_invalid_and_winsorize(df: pd.DataFrame, cap_iqr=True) -> pd.DataFrame:
    """
    Address obvious invalid Age entries and optionally winsorize Age using IQR bounds.
    - Replace Age == 0 with NaN (profile shows min Age = 0, likely invalid).
    - If cap_iqr is True, compute IQR-based lower/upper fences and clip Age into [lower, upper].
    Returns modified copy of df and preserves NaN for previously invalid ages.
    """
    df = df.copy()
    if 'Age' not in df.columns:
        return df
    # Ensure numeric
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    # Replace explicit zero (likely invalid) with NaN
    df.loc[df['Age'] == 0, 'Age'] = np.nan
    if cap_iqr:
        # compute IQR bounds on non-missing Age
        q1 = df['Age'].quantile(0.25)
        q3 = df['Age'].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        # Clip but keep NaNs as NaN
        df['Age'] = df['Age'].clip(lower=lower, upper=upper)
    return df

def drop_low_variance_categoricals(df: pd.DataFrame, threshold: float = 0.95) -> pd.DataFrame:
    """
    Drop categorical columns where the top category frequency >= threshold (default 95%).
    Based on the profile, candidate columns include Days_Policy_Claim, Days_Policy_Accident,
    WitnessPresent, AgentType, etc. This is conservative: columns are dropped only if dominance >= threshold.
    Returns modified copy of df.
    """
    df = df.copy()
    # consider object and category dtypes
    cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    to_drop = []
    for c in cat_cols:
        top_freq = df[c].value_counts(normalize=True, dropna=False).iloc[0] if not df[c].empty else 0
        if top_freq >= threshold:
            to_drop.append(c)
    if to_drop:
        df = df.drop(columns=to_drop)
    return df

def drop_high_corr_numeric(df: pd.DataFrame, threshold: float = 0.9) -> pd.DataFrame:
    """
    Identify pairs of numeric columns with absolute Pearson correlation >= threshold and drop the second column in each pair.
    This helps remove numeric columns that may leak identical information (profile shows PolicyNumber vs Year corr = 0.937).
    Returns modified copy of df.
    """
    df = df.copy()
    num = df.select_dtypes(include=[np.number])
    if num.shape[1] <= 1:
        return df
    corr = num.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] >= threshold)]
    # Drop only if present in original df (they are numeric so they are)
    if to_drop:
        df = df.drop(columns=to_drop)
    return df

def clean_pipeline(df: pd.DataFrame,
                   drop_id: bool = True,
                   fix_zeros: bool = True,
                   convert_to_category: bool = True,
                   cap_age: bool = True,
                   drop_low_variance: bool = True,
                   drop_high_corr: bool = True) -> pd.DataFrame:
    """
    Wrapper function that applies remediation helpers conditionally.
    Flags:
      - drop_id: drop PolicyNumber
      - fix_zeros: replace '0' placeholders in DayOfWeekClaimed and MonthClaimed with NaN
      - convert_to_category: convert object columns to category dtype (no encoding)
      - cap_age: replace Age==0 with NaN and winsorize Age using IQR
      - drop_low_variance: drop categorical columns dominated by a single level (>=95%)
      - drop_high_corr: drop numeric columns highly correlated (|corr| >= 0.9)
    Returns a cleaned DataFrame copy.
    """
    cleaned = df.copy()
    if drop_id:
        cleaned = drop_policy_number(cleaned)
    if fix_zeros:
        cleaned = replace_placeholder_zeros(cleaned)
    if convert_to_category:
        # do not convert 'MonthClaimed' / 'DayOfWeekClaimed' if they were replaced to NaN; conversion is harmless
        cleaned = convert_object_to_category(cleaned)
    if cap_age:
        cleaned = handle_age_invalid_and_winsorize(cleaned, cap_iqr=True)
    if drop_low_variance:
        cleaned = drop_low_variance_categoricals(cleaned, threshold=0.95)
    if drop_high_corr:
        cleaned = drop_high_corr_numeric(cleaned, threshold=0.9)
    return cleaned

# Example usage (single-line):
df_clean = clean_pipeline(df, drop_id=True, fix_zeros=True, convert_to_category=True, cap_age=True, drop_low_variance=True, drop_high_corr=True)

In [ ]:
TARGET_COL = "FraudFound_P"  # change for your dataset
df[TARGET_COL].value_counts()

In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train.shape, X_test.shape

Preprocessing pipeline

In [ ]:
num_cols = X_train.select_dtypes(exclude=["object","category"]).columns
cat_cols = X_train.select_dtypes(include=["object","category"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

num_cols[:15], cat_cols[:15]

In [ ]:
# ------------------------------------
# Decision Tree (Classification) + GridSearch (MCC) + Print Tree
# ------------------------------------

import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import matthews_corrcoef, make_scorer

# Stratified CV keeps class balance similar across splits
cv = StratifiedShuffleSplit(n_splits=10, test_size=0.2, random_state=42)

# MCC scorer (higher is better)
mcc_scorer = make_scorer(matthews_corrcoef)
# More positive is better, like another metric

# -------------------------------------------
# 1) Pipeline
# -------------------------------------------
pipe_dt = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42)) # lass not regression like a number
])

# -------------------------------------------
# 2) Small param grid (fast)
# -------------------------------------------
param_grid_dt = {
    "classifier__max_depth": [8, 15], #lets use a deeper tree to illustrate the need for Feature Importance
    "classifier__criterion": ["gini", "entropy", "log_loss"]
}

# -------------------------------------------
# 3) GridSearchCV
# -------------------------------------------
gs_dt = GridSearchCV(
    estimator=pipe_dt,
    param_grid=param_grid_dt,
    cv=cv,
    scoring=mcc_scorer,
    n_jobs=-1,
    verbose=1
)

# -------------------------------------------
# 4) Fit
# -------------------------------------------
gs_dt.fit(X_train, y_train)
print("Decision Tree grid search complete.")
print("Best DT Params:", gs_dt.best_params_)
print("Best CV MCC:", gs_dt.best_score_)

# -------------------------------------------
# 5) Test MCC
# -------------------------------------------
dt_best = gs_dt.best_estimator_
dt_pred = dt_best.predict(X_test)
print("\nTest MCC (Decision Tree):", matthews_corrcoef(y_test, dt_pred))

# -------------------------------------------
# 6) Print the tree rules (text)
# -------------------------------------------
# Get feature names after preprocessing (works for ColumnTransformer in recent sklearn)
pre = dt_best.named_steps["preprocessor"]
clf = dt_best.named_steps["classifier"]

try:
    feature_names = pre.get_feature_names_out()
except Exception:
    feature_names = [f"f{i}" for i in range(clf.n_features_in_)]

tree_text = export_text(clf, feature_names=list(feature_names), max_depth=6)
print("\nDecision Tree (first 6 levels):\n") # purposely keep to 6
print(tree_text)

In [ ]:
# -------------------------------------------
# Optional: plot tree (needs matplotlib)
# -------------------------------------------
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
plt.figure(figsize=(18, 10))
plot_tree(clf, feature_names=feature_names, class_names=True, filled=True, max_depth=4)
plt.show()

In [ ]:
# -------------------------------------------
# 7) Feature Importance (FI) from the best Decision Tree
# (use clear names so it won't clash with XGB variables)
# -------------------------------------------
import pandas as pd

dt_preproc = dt_best.named_steps["preprocessor"]
dt_clf = dt_best.named_steps["classifier"]

# Get feature names after preprocessing
try:
    dt_feature_names = dt_preproc.get_feature_names_out()
except Exception:
    dt_feature_names = [f"f{i}" for i in range(dt_clf.n_features_in_)]

# DecisionTreeClassifier has feature_importances_
dt_fi = pd.Series(dt_clf.feature_importances_, index=dt_feature_names)

# Sort high to low, and drop zeros for readability (optional)
dt_fi_sorted = dt_fi.sort_values(ascending=False)
dt_fi_nonzero = dt_fi_sorted[dt_fi_sorted > 0]

print("\nTop Feature Importances (Decision Tree):")
print(dt_fi_nonzero.head(20))

In [ ]:
# ------------------------------------
# 0. NOTE: This block takes quite a while to run, do it before moving onto explanation of code
# ------------------------------------

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# For classification, prefer stratified splits so class balance is similar in each split
cv = StratifiedShuffleSplit(n_splits=10, test_size=0.2, random_state=42)

# -------------------------------------------
# 1. Create pipelines for both models
# -------------------------------------------

pipe_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

pipe_xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        random_state=42,
        eval_metric="logloss"  # sensible default for binary classification
    ))
])

# -------------------------------------------
# 2. Define parameter grids
# Keep them small for speed and simplicity
# -------------------------------------------

param_grid_rf = {
    "classifier__n_estimators": [50, 200],
    "classifier__max_depth": [5, 10],
    "classifier__criterion": ["gini", "entropy", "log_loss"]
}

param_grid_xgb = {
    "classifier__n_estimators": [50, 200],
    "classifier__max_depth": [2, 4, 6],
    "classifier__eval_metric": ["logloss", "auc"]
}

# -------------------------------------------
# 3. Create GridSearchCV objects
# -------------------------------------------
#make scorer mcc
from sklearn.metrics import make_scorer, matthews_corrcoef


gs_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    cv=cv,
    scoring="matthews_corrcoef",
    n_jobs=-1,
    verbose=1
)

gs_xgb = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    cv=cv,
    scoring="matthews_corrcoef",
    n_jobs=-1,
    verbose=1
)

# -------------------------------------------
# 4. Fit both models
# (Students can run one at a time if needed)
# -------------------------------------------

gs_rf.fit(X_train, y_train)
print("Random Forest grid search complete.")

gs_xgb.fit(X_train, y_train)
print("XGBoost grid search complete.")

# -------------------------------------------
# 5. Evaluate on test set
# -------------------------------------------

from sklearn.metrics import classification_report, confusion_matrix, matthews_corrcoef

rf_pred = gs_rf.best_estimator_.predict(X_test)
xgb_pred = gs_xgb.best_estimator_.predict(X_test)

print("\nRF MCC:", matthews_corrcoef(y_test, rf_pred))
print("Best RF Params:", gs_rf.best_params_)
print("\nRF Classification Report:\n", classification_report(y_test, rf_pred))
print("RF Confusion Matrix:\n", confusion_matrix(y_test, rf_pred))

print("\nXGB MCC:", matthews_corrcoef(y_test, xgb_pred))
print("Best XGB Params:", gs_xgb.best_params_)
print("\nXGB Classification Report:\n", classification_report(y_test, xgb_pred))
print("XGB Confusion Matrix:\n", confusion_matrix(y_test, xgb_pred))

In [ ]:
# -------------------------------------------
# 6) Feature Importance (FI) for the best XGBoost model (from GridSearch)
# (use clear names so it won't clash with DT variables)
# -------------------------------------------
import numpy as np
import pandas as pd

xgb_best_pipe = gs_xgb.best_estimator_
xgb_preproc = xgb_best_pipe.named_steps["preprocessor"]
xgb_clf = xgb_best_pipe.named_steps["classifier"]

# Feature names after preprocessing
try:
    xgb_feature_names = xgb_preproc.get_feature_names_out()
except Exception:
    xgb_feature_names = np.array([f"f{i}" for i in range(xgb_clf.n_features_in_)], dtype=object)

# XGBoost importance types:
# - "gain": average gain of splits using the feature (closest to tree FI conceptually)
# - "weight": number of times feature is used in splits
# - "cover": average coverage (samples affected) of splits using the feature
xgb_importance_type = "gain"

# Get raw importances keyed by "f0", "f1", ...
xgb_raw = xgb_clf.get_booster().get_score(importance_type=xgb_importance_type)

# Convert to aligned Series for all features (missing -> 0)
xgb_fi = pd.Series(0.0, index=xgb_feature_names)
for k, v in xgb_raw.items():
    # k looks like "f12" -> index 12
    idx = int(k[1:])
    if idx < len(xgb_feature_names):
        xgb_fi.iloc[idx] = float(v)

xgb_fi_sorted = xgb_fi.sort_values(ascending=False)

print(f"\nTop XGBoost Feature Importances (importance_type='{xgb_importance_type}'):")
print(xgb_fi_sorted.head(20))

print("\nNotes:")
print("- XGBoost FI depends on importance_type (gain/weight/cover).")
print("- These values are not on the same scale as Decision Tree FI (Decision Tree sums to 1).")
print("- Adding FI numbers across different runs/folds does not make sense.")
print("- Comparing FI across different model families is not directly meaningful.")

# Optional: show other importance types for comparison (top 10 each)
for t in ["weight", "cover"]:
    xgb_raw_t = xgb_clf.get_booster().get_score(importance_type=t)

    xgb_fi_t = pd.Series(0.0, index=xgb_feature_names)
    for k, v in xgb_raw_t.items():
        idx = int(k[1:])
        if idx < len(xgb_feature_names):
            xgb_fi_t.iloc[idx] = float(v)

    print(f"\nTop XGBoost FI (importance_type='{t}'):")
    print(xgb_fi_t.sort_values(ascending=False).head(10))

In [ ]:
# -------------------------------------------
# 8) Permutation Feature Importance (pFI)
# What mattered most to prediction accuracy
# -------------------------------------------

import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import matthews_corrcoef, make_scorer

# Use MCC to stay consistent with model selection
mcc_scorer = make_scorer(matthews_corrcoef)

# IMPORTANT:
# - pFI must be computed on held-out data
# - the model must already be trained
# - we do NOT refit the model here

# Compute permutation importance on the TEST set
pfi = permutation_importance(
    estimator=dt_best,
    X=X_test,
    y=y_test,
    scoring=mcc_scorer,
    n_repeats=10,          # repeat permutations for stability
    random_state=42,
    n_jobs=-1
)

# Mean drop in score across repeats
pfi_mean = pd.Series(
    pfi.importances_mean,
    index=feature_names
)

# Optional: variability across repeats (for teaching discussion)
pfi_std = pd.Series(
    pfi.importances_std,
    index=feature_names
)

# Sort by importance (largest performance drop first)
pfi_sorted = pfi_mean.sort_values(ascending=False)

print("\nTop Permutation Feature Importances (Decision Tree, MCC drop):")
print(pfi_sorted.head(20))

In [ ]:
# Build comparison table
dt_fi_vs_pfi = pd.DataFrame({
    "DT_FI (split-based)": dt_fi,
    "DT_pFI (MCC_drop)": pfi_mean
})

# Sort by pFI (what matters most to accuracy)
dt_fi_vs_pfi_sorted = dt_fi_vs_pfi.sort_values(
    by="DT_pFI (MCC_drop)",
    ascending=False
)

print("\nDecision Tree: FI vs Permutation FI (sorted by pFI):")
print(dt_fi_vs_pfi_sorted.head(20))

Sharp

In [ ]:
import shap

# Get preprocessors and final estimators
dt_pre  = dt_best_pipe.named_steps["preprocessor"]
dt_clf  = dt_best_pipe.named_steps["classifier"]

xgb_pre = xgb_best_pipe.named_steps["preprocessor"]
xgb_clf = xgb_best_pipe.named_steps["classifier"]

def safe_feature_names(preprocessor):
    try:
        return list(preprocessor.get_feature_names_out())
    except Exception:
        return None

dt_feature_names  = safe_feature_names(dt_pre)
xgb_feature_names = safe_feature_names(xgb_pre)

print("DT feature names available:", dt_feature_names is not None)
print("XGB feature names available:", xgb_feature_names is not None)

In [ ]:
# Transform background and test sets
X_bg_dt  = dt_pre.transform(X_bg)
X_bg_xgb = xgb_pre.transform(X_bg)

X_test_dt  = dt_pre.transform(X_test)
X_test_xgb = xgb_pre.transform(X_test)

print("Shapes:")
print("X_bg_dt  :", X_bg_dt.shape)
print("X_test_dt:", X_test_dt.shape)

Global SHAP: summary plots (beeswarm + bar)
How to read beeswarm plots:

Each dot is one sample
x-axis is SHAP value
negative: pushes towards class 0
positive: pushes towards class 1
colour shows feature value (high vs low)
Bar plot is mean absolute SHAP (global importance, but direction removed).

In [ ]:
expl_dt = shap.TreeExplainer(dt_clf)
shap_dt = expl_dt.shap_values(X_test_dt)

if isinstance(shap_dt, list):
    shap_dt_c1 = shap_dt[1]
else:
    shap_dt_c1 = shap_dt[:, :, 1]

In [ ]:
from matplotlib import pyplot as plt

# SHAP beeswarm plot
used = np.where(dt_clf.feature_importances_ > 0)[0]

X_test_used = X_test_dt[:, used]
shap_used   = shap_dt_c1[:, used]
names_used  = [dt_feature_names[i] for i in used]

shap.summary_plot(
    shap_used,
    X_test_used,
    feature_names=names_used,
    show=False
)
plt.tight_layout()
plt.show()

In [ ]:
# XGBoost
expl_xgb = shap.TreeExplainer(xgb_clf)
shap_xgb = expl_xgb.shap_values(X_test_xgb)

if isinstance(shap_xgb, list) and len(shap_xgb) == 2:
    shap_xgb_c1 = shap_xgb[1]
else:
    shap_xgb_c1 = shap_xgb

print("XGB SHAP shape (class 1):", np.array(shap_xgb_c1).shape)

In [ ]:
# XGB beeswarm
shap.summary_plot(
    shap_xgb_c1,
    X_test_xgb,
    feature_names=xgb_feature_names,
    show=False
)
plt.tight_layout()
plt.show()

# XGB bar (mean |SHAP|)
shap.summary_plot(
    shap_xgb_c1,
    X_test_xgb,
    feature_names=xgb_feature_names,
    plot_type="bar",
    show=False
)
plt.tight_layout()
plt.show()

Why SHAP Beeswarm is Better (and Different) from FI and pFI Bar Plots
When you compare Feature Importance (FI), Permutation Feature Importance (pFI), and the SHAP beeswarm plot, you are looking at three different ways of answering:

“Which features matter most in this model?”

But doctors often need a more useful question:

“For this feature, should I be worried about a high value or a low value?”

Standard Definitions (What Each Plot Means)
Method	What it tells you (simple meaning)	Output Type
FI (Built-in importance)	Which features the model uses most during training	One bar per feature
pFI (Permutation importance)	Which features reduce accuracy when you shuffle them	One bar per feature
SHAP Beeswarm	How features push predictions higher or lower in the data	Many dots per feature
1. FI and pFI Only Tell You “worst perimeter is Important”
FI and pFI bar plots might show:

worst perimeter is one of the top important features.

But doctors still do not know:

Does a high worst perimeter increase cancer risk?
Does a low worst perimeter increase cancer risk?
Which direction is the warning sign?
Bar plots cannot answer this.

2. Beeswarm Shows Whether worst perimeter Pushes Risk Up or Down
The SHAP beeswarm plot shows direction:

Left side → feature pushes prediction lower
Right side → feature pushes prediction higher
So doctors can understand:

“Does worst perimeter push the model towards benign or malignant?”

Example:

If lower worst perimeter values push prediction towards malignant, then low values are the worrying sign.
FI and pFI cannot show this direction. Beeswarm can.

3. Beeswarm Shows the Pattern Across the Dataset
FI and pFI give only one bar:

worst perimeter → one importance score
But beeswarm shows many dots:

worst perimeter → many patients in the dataset
This helps doctors see:

Low worst perimeter values often push towards malignant
High worst perimeter values often push towards benign
So beeswarm answers:

“Is this pattern happening across the dataset?”

4. Beeswarm Uses Colour to Show High vs Low worst perimeter
In the beeswarm plot:

Red = high worst perimeter values
Blue = low worst perimeter values
This makes it very clear:

If blue dots are mostly on the right, then low worst perimeter increases malignant risk
Doctors can directly see:

“Low values are more alarming in this model.”

FI and pFI cannot show this.

5. Beeswarm Can Show When worst perimeter is Not Always Simple
Sometimes you may see:

Blue dots on both left and right sides
This means:

Low worst perimeter is often a warning sign, but not always.

Doctors should not rely on one feature alone.

Bar plots hide this kind of detail.

Known Caveats and Exceptions
Method	Main limitation
FI	Can be biased towards features with many split points
pFI	Can look weak when features are strongly related
SHAP	Takes more time to compute and needs careful reading
Important note:

If another feature is very similar to worst perimeter, SHAP may split the effect.
Summary: Why Beeswarm is Often Better
SHAP beeswarm is usually preferred because it shows:

✅ worst perimeter matters

✅ whether it pushes prediction towards malignant or benign

✅ whether high (red) or low (blue) values are the warning sign

✅ clearer patterns across the dataset

So instead of only saying:

“worst perimeter is important”

Beeswarm explains:

“worst perimeter is important, and in this model, lower values (blue) push predictions towards malignant, so doctors should pay attention to low values.”

In [ ]:
xgb_feature_names[:]

In [ ]:
import numpy as np

# Mean absolute SHAP per feature
mean_abs_shap = np.abs(shap_xgb_c1).mean(axis=0)

# Sort feature indices by impact (descending)
order = np.argsort(mean_abs_shap)[::-1]

# Sorted feature names (most impactful first)
xgb_feature_names_sorted = [xgb_feature_names[i] for i in order]

# Optional: preview top features
print("Top SHAP features:")
for name, val in zip(xgb_feature_names_sorted[:10], mean_abs_shap[order][:10]):
    print(f"{name:30s}  mean|SHAP| = {val:.4f}")

In [ ]:
# Pick a feature by name (copy from your SHAP bar plot labels)
# Tip: try xgb_feature_names[:10] to preview candidates
feature_name = xgb_feature_names_sorted[0]
print("Using feature:", feature_name)

shap.dependence_plot(
    feature_name,
    shap_xgb_c1,
    X_test_xgb,
    feature_names=xgb_feature_names,
    interaction_index=None,   # <-- removes colour dimension
    show=False
)
plt.tight_layout()
plt.show()

How to Read This SHAP Dependence Plot (worst perimeter)
This plot helps doctors understand how worst perimeter affects the model prediction.

A doctor’s real question is:

“At what value should I start to be more careful?”

1) X-axis: Tumour perimeter value
The bottom axis shows the actual value of:

num__worst perimeter

Left side = smaller perimeter
Right side = larger perimeter
2) Y-axis: Does the model become more worried or less worried?
The SHAP value shows the impact:

SHAP above 0 → pushes prediction towards malignant
SHAP below 0 → pushes prediction towards benign
So this tells doctors:

“Does this perimeter value increase cancer risk or reduce it?”

3) Actionable pattern in this plot
When perimeter is small (around 60–100)
Most points are above 0.

This means:

Small worst perimeter values push the model towards malignant.

So low values are a warning sign.

When perimeter is larger (above ~110)
Most points are below 0.

This means:

Larger worst perimeter values push the model towards benign.

So the model is less worried.

4) The overlap region (around 100–110)
Notice that around 100 to 110, points start to mix.

This means:

This is a transition zone where the model is not perfectly certain.

Some cases still look risky even near 100.

5) Healthcare recommendation (false negatives are costly)
In healthcare:

A false negative (missing a malignant case) is more harmful to patients
Doctors prefer to be more cautious
So instead of using a cut-off like 100, it may be safer to recommend:

Doctors pay closer attention when worst perimeter is below 110

This reduces the chance of missing a risky patient in the overlap region.

Key takeaway
This plot gives an actionable insight:

worst perimeter is not just important, but values below about 110 should raise more concern, especially when patient safety is the priority.

In [ ]:
for feature_name in xgb_feature_names_sorted[:10]:
    shap.dependence_plot(
        feature_name,
        shap_xgb_c1,
        X_test_xgb,
        feature_names=xgb_feature_names,
        interaction_index=None,
        show=False
    )
    plt.tight_layout()
    plt.show()

Local explanations: waterfall plot (one case)
Why this matters: This is the most “stakeholder friendly” explanation.

You will explain one test case:

baseline (expected value)
top positive contributors
top negative contributors
final model output
For multiclass, you must decide which class you are explaining.

In [ ]:
import numpy as np
import shap
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 0) Student choices
# ------------------------------------------------------------
target_class = 1                       # choose 0 or 1 (true label group)
feature_name = xgb_feature_names_sorted[0]

# Choose how to pick the "most interesting" row for this feature:
# - "max_pos"  : row where SHAP(feature) is most positive (pushes UP most)
# - "max_neg"  : row where SHAP(feature) is most negative (pushes DOWN most)
# - "max_abs"  : row where |SHAP(feature)| is largest (strongest impact either way)
# - "near_zero": row where SHAP(feature) is closest to 0 (feature has least impact)
pick_mode = "max_pos"                  # change to: "max_neg", "max_abs", "near_zero"


# ------------------------------------------------------------
# Helper: get SHAP matrix for a chosen class (robust to formats)
# ------------------------------------------------------------
def get_shap_matrix_for_class(explainer, X, target_class: int):
    sv = explainer.shap_values(X)

    # Case A: list of arrays, one per class
    if isinstance(sv, list):
        return sv[target_class], explainer.expected_value[target_class]

    sv = np.asarray(sv)

    # Case B: 3D array (n_samples, n_features, n_classes)
    if sv.ndim == 3:
        base = explainer.expected_value
        base_for_class = base[target_class] if isinstance(base, (list, np.ndarray)) else base
        return sv[:, :, target_class], base_for_class

    # Case C: 2D array (n_samples, n_features) for binary
    # Treat this as the single output explanation (often margin / log-odds)
    base = explainer.expected_value
    base_for_class = base if not isinstance(base, (list, np.ndarray)) else base[0]
    return sv, base_for_class


# ------------------------------------------------------------
# 1) Get SHAP for the chosen class (or single-output fallback)
# ------------------------------------------------------------
shap_for_class, base_for_class = get_shap_matrix_for_class(expl_xgb, X_test_xgb, target_class)

print("shap_for_class shape:", np.asarray(shap_for_class).shape)
print("baseline (base_for_class):", base_for_class)


# ------------------------------------------------------------
# 2) Filter row indices by TRUE label group
# ------------------------------------------------------------
y_test_arr = np.asarray(y_test)
class_idx = np.where(y_test_arr == target_class)[0]
assert len(class_idx) > 0, f"No rows found in y_test for target_class={target_class}"


# ------------------------------------------------------------
# 3) Choose row based on pick_mode for the selected feature
# ------------------------------------------------------------
feat_j = xgb_feature_names.index(feature_name)

shap_vals_feat_in_class = shap_for_class[class_idx, feat_j]

if pick_mode == "max_pos":
    # Most positive SHAP value
    best_pos_in_class = int(np.argmax(shap_vals_feat_in_class))
elif pick_mode == "max_neg":
    # Most negative SHAP value
    best_pos_in_class = int(np.argmin(shap_vals_feat_in_class))
elif pick_mode == "max_abs":
    # Largest absolute SHAP value
    best_pos_in_class = int(np.argmax(np.abs(shap_vals_feat_in_class)))
elif pick_mode == "near_zero":
    # Closest to zero (least impact)
    best_pos_in_class = int(np.argmin(np.abs(shap_vals_feat_in_class)))
else:
    raise ValueError('pick_mode must be one of: "max_pos", "max_neg", "max_abs", "near_zero"')

i = int(class_idx[best_pos_in_class])  # row index in X_test

print(f"\nChosen true target_class: {target_class}")
print(f"Chosen feature: {feature_name}")
print(f"Row selection mode: {pick_mode}")
print(f"Selected row index i (within X_test): {i}")
print(f"Feature SHAP at i: {shap_for_class[i, feat_j]:.6f}")


# ------------------------------------------------------------
# 4) Waterfall for that row
# ------------------------------------------------------------
exp = shap.Explanation(
    values=shap_for_class[i],
    base_values=base_for_class,
    data=X_test_xgb[i],
    feature_names=xgb_feature_names
)

# Top contributors (good for students)
vals = np.asarray(exp.values)
top_idx = np.argsort(np.abs(vals))[::-1][:12]
print("\nTop contributors for this case (by |SHAP|):")
for k in top_idx:
    direction = "pushes UP" if vals[k] > 0 else "pushes DOWN"
    print(f"- {xgb_feature_names[k]:30s} SHAP={vals[k]: .4f} ({direction})")

shap.plots.waterfall(exp, max_display=12)
plt.show()

SHAP vs FI vs pFI: quick comparison for action
What you are comparing
pFI answers: “What hurts MCC if this column becomes unreliable?”
SHAP answers: “What pushes predictions most, and in which direction?”
Important mismatch:
pFI is computed on raw columns. SHAP often works on expanded columns (after one-hot).

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer, matthews_corrcoef

mcc_scorer = make_scorer(matthews_corrcoef)

# pFI on full pipeline (permutes raw columns correctly)
pfi_xgb = permutation_importance(
    estimator=xgb_best_pipe,
    X=X_test,
    y=y_test,
    scoring=mcc_scorer,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

pfi_series = pd.Series(pfi_xgb.importances_mean, index=X_test.columns).sort_values(ascending=False)

# SHAP global importance on transformed feature space
shap_global = pd.Series(np.abs(shap_xgb_c1).mean(axis=0), index=xgb_feature_names).sort_values(ascending=False)

print("Top raw-column pFI (what hurts MCC if broken):")
display(pfi_series.head(10))

print("Top transformed-feature SHAP (what pushes predictions most):")
display(shap_global.head(10))

Decision plot (optional but powerful)
From FI, pFI, and the SHAP bar plot, you saw which predictors are generally important.

In the SHAP beeswarm plot, you saw how each predictor behaves across many cases (some cases pushed up, some pushed down).

Now, you can piece them together to understand how the model gets from the baseline to the final prediction for each case.

For this decision plot, you are looking at many cases at once, so it shows the “paths” of predictions for a small group of samples.

How to read it (on this plot)
Each line = one case (one row).

The line starts near the baseline (around the grey vertical line, about 0).

As you move up the feature list, the prediction is updated step by step.

A big sideways jump at a feature means that feature had a big impact for that case:

jump right → pushes the model output higher
jump left → pushes the model output lower
Where the line ends at the top is the final model output for that case.

What this specific plot is telling you
You can see two clear groups:

Blue lines end far on the left (about −6 to −8): these cases are being pushed strongly towards the lower-output class.
Pink/red lines end far on the right (about +6 to +10): these cases are being pushed strongly towards the higher-output class.
The biggest “separation” happens near the top features, especially:

num__worst perimeter
num__worst texture
num__area error
num__worst concave points
num__worst area
num__worst radius
Those features are doing most of the heavy lifting in moving cases left vs right.

Daily life analogy
Think of it like building your final shopping bill:

you start from a starting total (baseline),
each item (feature) adds or subtracts,
a big price item causes a big jump,
the final total at the end is your final bill (final prediction).
If you tell me which single line is “your case” (e.g., its index in the sample list), I can explain that one path feature-by-feature (which features pushed it right, which pushed it left, and the biggest turning points).

Tip: keep the sample size small for readability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import shap

# Decision plot for first 30 samples
idx = list(range(30))

# Get a baseline value (expected_value) safely
base_xgb = expl_xgb.expected_value
if isinstance(base_xgb, (list, np.ndarray)):
    # binary or multiclass case
    base_value = base_xgb[target_class] if isinstance(base_xgb, (list, np.ndarray)) else base_xgb
else:
    # scalar baseline
    base_value = base_xgb

shap.decision_plot(
    base_value,
    shap_xgb_c1[idx],
    feature_names=xgb_feature_names,
    show=False
)
plt.tight_layout()
plt.show()

In [ ]:
Wrap-up prompts (write-up practice)
Answer briefly in markdown:

Which 3–5 signals are strongest, and do they make domain sense?
For the top 2 signals, what feature values push risk up vs down?
Propose one low-risk action per signal (what would you do differently?)
Which signals might be unstable (correlation, leakage suspicion, proxy features)?
In 2–3 sentences: why is SHAP more decision-ready than FI and pFI?

In [ ]:
import numpy as np
import pandas as pd

def summarise_dependence_as_text(
    feature_name: str,
    shap_values_2d,          # shap_xgb_c1: (n_samples, n_features)
    X_2d,                    # X_test_xgb: (n_samples, n_features)
    feature_names: list,
    critical_q=0.90,         # top 10% by |SHAP|
    bins=10                  # for a simple binned trend
):
    j = feature_names.index(feature_name)
    x = np.asarray(X_2d)[:, j]
    s = np.asarray(shap_values_2d)[:, j]

    # 1) Direction (simple correlation as a sign)
    corr = np.corrcoef(x, s)[0, 1]
    direction = "increases" if corr > 0 else "decreases"

    # 2) Turning point (approx where SHAP crosses 0)
    # If it never crosses, report "no crossing"
    sign_change = np.where(np.sign(s[:-1]) != np.sign(s[1:]))[0]
    if len(sign_change) > 0:
        # pick the crossing nearest to s=0 by local min |s|
        k = sign_change[np.argmin(np.abs(s[sign_change]))]
        turning_point = float(np.mean([x[k], x[k+1]]))
        turning_note = f"SHAP crosses 0 around x ≈ {turning_point:.3f}."
    else:
        turning_note = "SHAP does not clearly cross 0 in this sample (mostly one-sided impact)."

    # 3) Critical zone by |SHAP|
    thr = float(np.quantile(np.abs(s), critical_q))
    mask = np.abs(s) >= thr
    if mask.any():
        crit_min = float(np.min(x[mask]))
        crit_max = float(np.max(x[mask]))
        crit_note = (
            f"Critical impact (top {int((1-critical_q)*100)}% by |SHAP|): "
            f"|SHAP| ≥ {thr:.3f} occurs when x is roughly in [{crit_min:.3f}, {crit_max:.3f}]."
        )
    else:
        crit_note = "No points exceed the chosen critical threshold (unexpected)."

    # 4) Typical vs extreme magnitude
    mean_abs = float(np.mean(np.abs(s)))
    max_abs  = float(np.max(np.abs(s)))
    mag_note = f"Average |SHAP| ≈ {mean_abs:.3f}; max |SHAP| ≈ {max_abs:.3f}."

    # 5) Simple binned trend (helps describe ‘when it becomes critical’)
    # bin by feature quantiles for robustness
    qs = np.quantile(x, np.linspace(0, 1, bins + 1))
    # make bins unique (in case of repeated values)
    qs = np.unique(qs)
    if len(qs) >= 3:
        bin_ids = np.digitize(x, qs[1:-1], right=True)
        bin_summary = []
        for b in range(bin_ids.min(), bin_ids.max() + 1):
            xb = x[bin_ids == b]
            sb = s[bin_ids == b]
            if len(xb) == 0:
                continue
            bin_summary.append({
                "x_range": f"[{np.min(xb):.3f}, {np.max(xb):.3f}]",
                "mean_shap": float(np.mean(sb)),
                "mean_abs_shap": float(np.mean(np.abs(sb))),
                "n": int(len(xb))
            })
        bin_df = pd.DataFrame(bin_summary).sort_values("x_range")
        # take the 3 bins with largest mean_abs_shap
        top_bins = bin_df.sort_values("mean_abs_shap", ascending=False).head(3)
        bins_note = "Most impactful value ranges (by mean |SHAP|):\n" + top_bins.to_string(index=False)
    else:
        bins_note = "Binned trend unavailable (feature has too few unique values)."

    # Final text blob (student-friendly, API-friendly)
    text = (
        f"Dependence summary for feature: {feature_name}\n"
        f"- Overall trend: as {feature_name} increases, SHAP contribution generally {direction} "
        f"(corr ≈ {corr:.3f}).\n"
        f"- {turning_note}\n"
        f"- {crit_note}\n"
        f"- {mag_note}\n"
        f"- {bins_note}\n"
        f"Caveat: This is model-driven association, not proof of causation."
    )

    return text

# Example usage
feature_name = xgb_feature_names_sorted[0]
summary_text = summarise_dependence_as_text(
    feature_name=feature_name,
    shap_values_2d=shap_xgb_c1,
    X_2d=X_test_xgb,
    feature_names=xgb_feature_names,
    critical_q=0.90,
    bins=10
)

print(summary_text)

In [ ]:
from google.colab import userdata
from openai import OpenAI

# Load key from Google Colab Secrets
api_key = userdata.get('OPENAI_API_KEY')

client = OpenAI(
    api_key=api_key,
)

In [ ]:
import numpy as np

TOP_N = 8  #typically 3–5 is ideal for students

mean_abs_shap = np.abs(shap_xgb_c1).mean(axis=0)
order = np.argsort(mean_abs_shap)[::-1]
top_feature_names = [xgb_feature_names[i] for i in order[:TOP_N]]

print("Top features by mean |SHAP|:")
for f in top_feature_names:
    print("-", f)

In [ ]:
dependence_summaries = []

for fname in top_feature_names:
    txt = summarise_dependence_as_text(
        feature_name=fname,
        shap_values_2d=shap_xgb_c1,
        X_2d=X_test_xgb,
        feature_names=xgb_feature_names,
        critical_q=0.90,
        bins=10
    )
    dependence_summaries.append(txt)

In [ ]:
payload_text = f"""
You are supporting decision-making using a trained XGBoost binary classification model.

IMPORTANT CONTEXT:
- Interpretations are based on SHAP values (model-driven, not causal).
- SHAP values are on the model output scale.
- Recommendations must consider MULTIPLE features together.

TASK:
1) Based on the SHAP evidence below, give 3 data-driven decision recommendations.
2) Each recommendation must reference at least TWO features.
3) State one risk or limitation per recommendation.
4) Propose 2 monitoring checks to detect drift or instability.

GLOBAL CONTEXT:
Top {TOP_N} features selected by mean absolute SHAP impact.

FEATURE-LEVEL DEPENDENCE SUMMARIES
(Each summary is auto-generated from SHAP dependence data):

{"\n\n".join(dependence_summaries)}

OUTPUT RULES:
- Do not treat any single feature as causal.
- Use short bullet points.
- Explicitly mention interactions or reinforcement between features.
""".strip()

print(payload_text)

In [ ]:
response = client.responses.create(
    model="gpt-5-mini",
    instructions="""
You are an expert data scientist working with medical expertise.
You understand tree-based models and SHAP explanations.
You must reason strictly from the provided SHAP evidence and dataset context.
You may draw reasonable conclusion and assumption from the context provided.
""",
    input=f"""
Dataset context:
- Dataset: scikit-learn Breast Cancer dataset
- Task: Binary classification (malignant vs benign)
- Model: XGBoost
- Interpretation method: SHAP (model-driven, log-odds scale)

Model interpretation evidence:
{payload_text}

Questions:

1. From a clinical decision-support perspective, how should recommendations be formed when using the model.

2. Based on the SHAP evidence across the important features, identify the strongest combined risk patterns the model appears to use, for clinician decision-making,
especially for the data ranges and combinations where training data is scarce.

3. Propose 3 decision-support recommendations that a medical expert could reasonably make using this model output.

4. Suggest 2 post-deployment monitoring checks to ensure these SHAP-based patterns remain stable and clinically sensible over time.

Rules:
- Use language to help non medically trained personels to understand.
- Treat the model as an aid to expert judgement, not a replacement.
- Base all reasoning on the SHAP evidence provided.
"""
)

print(response.output_text)